In [1]:
import xarray as xr
import pandas as pd
from scipy import stats
import numpy as np
from cartopy import crs as ccrs, feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.pyplot as plt
from metpy.units import units
import matplotlib.colors as colors
import metpy.calc as mpcalc
%run ../Feedback/Pendergrass-function.ipynb

ERROR 1: PROJ: proj_create_from_database: Open of /knight/anaconda_aug22/envs/aug22_env/share/proj failed


In [2]:
def T_feedback(dts,dta):
    #=======================read the kernel=========================
    diri='/zhoulab_rit/lzhuo/data/Pendergrass_kernel/'
    #---------------first, read the kernel-------------------------
    filename='ts.kernel.nc'
    fi=xr.open_dataset(diri+filename)
    #-------------fullsky--------------------------------------------
    kernel=fi.FLNT #12*192*288
    ts_kernel=preprocess_2D(kernel)
    #-------------clear sky-----------------------------------------
    kernel=fi.FLNTC #12*192*288
    ts_kernel_cs=preprocess_2D(kernel)

    filename='t.kernel.plev.nc'
    fi=xr.open_dataset(diri+filename)
    #---------------full sky--------------------------------------
    kernel=fi.FLNT #12*17*192*288
    ta_kernel=preprocess_3D(kernel)
    #---------------clear sky----------------------------------------
    kernel=fi.FLNTC #12*17*192*288
    ta_kernel_cs=preprocess_3D(kernel)
    
    #=========Regrid the dta and dts onto the coordinate of the kernel====
    new_lat=ta_kernel.lat
    new_lon=ta_kernel.lon
    dts=dts.interp(lat=new_lat,lon=new_lon,kwargs={"fill_value": "extrapolate"})
    dta=dta.interp(lat=new_lat,lon=new_lon,kwargs={"fill_value": "extrapolate"})
    
    #==========preprocess the data=======================================
    dts=preprocess_2D(dts)
    dta=preprocess_3D(dta)
    
    #===========do the calculation=======================================
    dLW_t   =temperature_feedback(ts_kernel,ta_kernel,dts,dta)
    dLW_t_cs=temperature_feedback(ts_kernel_cs,ta_kernel_cs,dts,dta)
    return dLW_t,dLW_t_cs